# Training YOLO on Fire Detection for Far-Field Scenario

## Overview
In this notebook, we train a **YOLOv8** model on a custom dataset designed for detecting **fire scenarios** in **far-field images**. This dataset represents one of the specific scenarios for our **Mixture of Experts (MoE)** model, where each expert specializes in a different scenario (e.g., fire detection in outdoor, indoor, satellite, or far-field environments).

### Key Steps:
- **Dataset**: The model is trained using a **far-field fire detection dataset**, which consists of images with fire-related features captured in distant or large-scale scenarios.
- **YOLOv8 Training**: The YOLOv8 model is fine-tuned on this dataset, learning to identify fire-related objects.
- **Scenario Expert**: This trained model acts as an expert specifically for detecting fire in far-field images and is part of a broader MoE-based approach for multi-scenario detection.



##  Data Loading & Preprocessing


In [1]:
from datasets import load_dataset

# Load the Pyro-SDIS dataset from Hugging Face
ds = load_dataset("pyronear/pyro-sdis")


C:\Users\pc\AppData\Roaming\Python\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import random

# Set a seed for reproducibility
random.seed(42)

# Select 1000 random samples from train
train_subset = ds['train'].shuffle(seed=42).select(range(1000))

# Select 100 random samples from validation
val_subset = ds['val'].shuffle(seed=42).select(range(100))


In [4]:
import os
from PIL import Image

# Base directory
base_dir = "pyro_subset_yolo"
os.makedirs(base_dir, exist_ok=True)

# Helper function to save images and labels
def save_subset_to_yolo_format(subset, split):
    img_dir = os.path.join(base_dir, "images", split)
    label_dir = os.path.join(base_dir, "labels", split)
    os.makedirs(img_dir, exist_ok=True)
    os.makedirs(label_dir, exist_ok=True)

    for item in subset:
        # Save image
        image = item["image"]
        img_path = os.path.join(img_dir, item["image_name"])
        image.save(img_path)

        # Save annotation (single line YOLO format)
        label_path = os.path.join(label_dir, item["image_name"].replace(".jpg", ".txt"))
        with open(label_path, "w") as f:
            f.write(item["annotations"])

    print(f"Saved {split} set to {img_dir} and {label_dir}")

# Save both subsets
save_subset_to_yolo_format(train_subset, "train")
save_subset_to_yolo_format(val_subset, "val")


Saved train set to pyro_subset_yolo\images\train and pyro_subset_yolo\labels\train
Saved val set to pyro_subset_yolo\images\val and pyro_subset_yolo\labels\val


In [6]:
import os

# Path to your labels directory
labels_dir = "C:/Users/pc/Desktop/DL/Project/FarField/pyro_subset_yolo/labels/train"

# Iterate through label files
for label_file in os.listdir(labels_dir):
    label_path = os.path.join(labels_dir, label_file)

    if os.path.isfile(label_path) and label_file.endswith(".txt"):
        with open(label_path, 'r') as file:
            lines = file.readlines()

        # Replace class 1 with class 0
        lines = [line.replace('1 ', '0 ') for line in lines]

        # Save the modified file
        with open(label_path, 'w') as file:
            file.writelines(lines)

        print(f"Updated {label_file} to class 0")


Updated force-06_cabanelle-125_2024-01-04T08-34-18.txt to class 0
Updated force-06_cabanelle-125_2024-01-08T13-12-25.txt to class 0
Updated force-06_cabanelle-125_2024-02-21T07-53-21.txt to class 0
Updated force-06_cabanelle-125_2024-02-21T07-54-51.txt to class 0
Updated force-06_cabanelle-125_2024-02-21T08-29-22.txt to class 0
Updated force-06_cabanelle-125_2024-02-21T08-30-23.txt to class 0
Updated force-06_cabanelle-125_2024-02-21T08-36-53.txt to class 0
Updated force-06_cabanelle-125_2024-02-27T13-31-25.txt to class 0
Updated force-06_cabanelle-125_2024-02-27T13-48-25.txt to class 0
Updated force-06_cabanelle-125_2024-02-27T14-11-26.txt to class 0
Updated force-06_cabanelle-125_2024-02-27T14-18-56.txt to class 0
Updated force-06_cabanelle-125_2024-02-27T14-45-58.txt to class 0
Updated force-06_cabanelle-125_2024-02-27T14-46-27.txt to class 0
Updated force-06_cabanelle-125_2024-02-27T14-48-57.txt to class 0
Updated force-06_cabanelle-125_2024-02-28T10-35-52.txt to class 0
Updated fo

In [7]:
import os

# Path to your labels directory
labels_dir = "C:/Users/pc/Desktop/DL/Project/FarField/pyro_subset_yolo/labels/val"

# Iterate through label files
for label_file in os.listdir(labels_dir):
    label_path = os.path.join(labels_dir, label_file)

    if os.path.isfile(label_path) and label_file.endswith(".txt"):
        with open(label_path, 'r') as file:
            lines = file.readlines()

        # Replace class 1 with class 0
        lines = [line.replace('1 ', '0 ') for line in lines]

        # Save the modified file
        with open(label_path, 'w') as file:
            file.writelines(lines)

        print(f"Updated {label_file} to class 0")


Updated force-06_cabanelle-125_2024-02-24T09-07-28.txt to class 0
Updated force-06_cabanelle-244_2024-09-19T14-03-44.txt to class 0
Updated force-06_cabanelle-327_2024-01-01T16-49-40.txt to class 0
Updated force-06_cabanelle-327_2024-02-23T08-55-55.txt to class 0
Updated force-06_cabanelle-327_2024-04-02T18-49-16.txt to class 0
Updated force-06_cabanelle-327_2024-04-02T18-59-14.txt to class 0
Updated force-06_cabanelle-327_2024-04-04T19-29-52.txt to class 0
Updated force-06_courmettes-160_2024-01-08T12-56-37.txt to class 0
Updated force-06_courmettes-160_2024-02-13T12-56-15.txt to class 0
Updated force-06_courmettes-160_2024-02-13T14-10-16.txt to class 0
Updated force-06_courmettes-160_2024-04-03T08-36-45.txt to class 0
Updated force-06_courmettes-160_2024-04-03T09-51-07.txt to class 0
Updated force-06_courmettes-160_2024-04-03T09-56-17.txt to class 0
Updated force-06_courmettes-160_2024-04-03T10-22-47.txt to class 0
Updated force-06_courmettes-160_2024-04-03T11-33-33.txt to class 0
Up

##  Training YOLOv8 on Far-Field Fire Detection

We utilize the **YOLOv8** object detection model to train on a custom dataset defined in `farfield.yaml`. The configuration for training is as follows:

- 🧠 **Model**: `yolov8n.pt` (Nano variant — optimized for speed and prototyping)
- 📁 **Dataset**: Defined in `farfield.yaml`
- 🖼️ **Image Size**: 640×640
- 🔁 **Epochs**: 100
- 📦 **Batch Size**: 16
- 💻 **Device**: GPU (device 0)

This setup helps the model learn to detect **fire in far-field scenarios**, which often includes distant flames or smoke in wide-view environments.


In [8]:
import torch
print("GPU available:", torch.cuda.is_available())
print("Current device:", torch.cuda.get_device_name(0))

GPU available: True
Current device: NVIDIA GeForce RTX 3070


In [ ]:
# ------------------------ Imports ------------------------
import os, cv2, torch, numpy as np, matplotlib.pyplot as plt
import albumentations as A
from timm import create_model
from tqdm import tqdm
from albumentations.pytorch import ToTensorV2
from sklearn.metrics import precision_score, recall_score, f1_score
from torch.utils.data import Dataset, DataLoader
from torchvision.models.detection import FasterRCNN
from torchvision.models.detection.rpn import AnchorGenerator
from torchvision.ops import box_iou, MultiScaleRoIAlign
from torchvision.models.detection.image_list import ImageList
import torch.nn as nn

# ------------------------ Dataset ------------------------
class FireDetectionDataset(Dataset):
    def __init__(self, images_dir, labels_dir, transform=None):
        self.images_dir = images_dir
        self.labels_dir = labels_dir
        self.image_files = [f for f in os.listdir(images_dir) if f.endswith(('.jpg', '.png'))]
        self.transform = transform

    def __getitem__(self, idx):
        img_path = os.path.join(self.images_dir, self.image_files[idx])
        label_path = os.path.join(self.labels_dir, os.path.splitext(self.image_files[idx])[0] + '.txt')

        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        h, w, _ = image.shape

        boxes, labels = [], []
        if os.path.exists(label_path):
            with open(label_path) as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) != 5:
                        continue
                    cls, cx, cy, bw, bh = map(float, parts)
                    x1 = (cx - bw / 2) * w
                    y1 = (cy - bh / 2) * h
                    x2 = (cx + bw / 2) * w
                    y2 = (cy + bh / 2) * h
                    boxes.append([x1, y1, x2, y2])
                    labels.append(1)

        if not boxes:
            boxes = [[0, 0, 1, 1]]
            labels = [0]

        if self.transform:
            augmented = self.transform(image=image, bboxes=boxes, labels=labels)
            image = augmented['image']
            boxes = augmented['bboxes']
            labels = augmented['labels']

        target = {
            'boxes': torch.tensor(boxes, dtype=torch.float32),
            'labels': torch.tensor(labels, dtype=torch.int64)
        }
        return image, target

    def __len__(self):
        return len(self.image_files)

# ------------------------ Swin Backbone ------------------------
class SwinBackbone(nn.Module):
    def __init__(self, out_channels=256):
        super().__init__()
        self.body = create_model('swin_tiny_patch4_window7_224', pretrained=True, features_only=True)
        self.fpn_input_channels = [96, 192, 384, 768]
        self.lateral_convs = nn.ModuleList([
            nn.Conv2d(in_c, out_channels, kernel_size=1) for in_c in self.fpn_input_channels
        ])

    def forward(self, x):
        feats = self.body(x)
        fpn_feats = {}
        for idx, feat in enumerate(feats):
            if feat.dim() == 4 and feat.shape[-1] == self.fpn_input_channels[idx]:
                feat = feat.permute(0, 3, 1, 2)
            fpn_feats[str(idx)] = self.lateral_convs[idx](feat)
        return fpn_feats

class IdentityTransform(nn.Module):
    def forward(self, images, targets=None):
        batch_images = torch.stack(images) if isinstance(images, list) else images
        image_sizes = [(224, 224) for _ in range(batch_images.shape[0])]
        return ImageList(batch_images, image_sizes), targets
    def postprocess(self, result, image_shapes, original_image_sizes):
        return result

# ------------------------ Model Builder ------------------------
def get_swin_frcnn(num_classes=2):
    backbone = SwinBackbone(out_channels=256)
    anchor_gen = AnchorGenerator(
        sizes=((16, 32), (32, 64), (64, 128), (128, 256)),
        aspect_ratios=((0.5, 1.0, 2.0),) * 4
    )
    roi_pooler = MultiScaleRoIAlign(["0", "1", "2", "3"], output_size=7, sampling_ratio=2)
    model = FasterRCNN(backbone, num_classes, anchor_generator=anchor_gen, box_roi_pool=roi_pooler, transform=None)
    model.transform = IdentityTransform()
    return model

# ------------------------ Preprocessing ------------------------
transform = A.Compose([
    A.Resize(224, 224),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
], bbox_params=A.BboxParams(format='pascal_voc', label_fields=['labels']))

# ------------------------ Loaders ------------------------
train_dataset = FireDetectionDataset("FarField/train/images", "FarField/train/labels", transform)
val_dataset = FireDetectionDataset("FarField/valid/images", "FarField/valid/labels", transform)
test_dataset = FireDetectionDataset("FarField/test/images", "FarField/test/labels", transform)

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, collate_fn=lambda x: tuple(zip(*x)))
val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False, collate_fn=lambda x: tuple(zip(*x)))
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False, collate_fn=lambda x: tuple(zip(*x)))

# ------------------------ Train ------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = get_swin_frcnn(num_classes=2).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)

print("📦 Training FarField Expert...")
model.train()
for epoch in range(10):
    total_loss = 0
    for batch_idx, (images, targets) in enumerate(tqdm(train_loader)):
        try:
            images = torch.stack(images).to(device)
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
            loss_dict = model(images, targets)
            loss = sum(loss for loss in loss_dict.values())
            optimizer.zero_grad(); loss.backward(); optimizer.step()
            total_loss += loss.item()
        except Exception as e:
            print(f"⚠️ Batch {batch_idx} skipped: {e}")
    print(f"Epoch {epoch+1} completed - Avg Loss: {total_loss:.4f}")

torch.save(model.state_dict(), "swin_frcnn_farfield.pt")
print("✅ Saved as swin_frcnn_farfield.pt")




In [11]:
import random

# Set a seed for reproducibility
random.seed(42)

# Select 50 random samples from the validation set
test_subset = ds['val'].shuffle(seed=42).select(range(50))

# If you want to save the test set, for instance as image paths and annotations
test_images = test_subset['image']
test_annotations = test_subset['annotations']

# You can print or inspect the selected samples
print(f"Test set size: {len(test_images)}")


Test set size: 50


In [16]:
import os
import shutil
from PIL import Image

# Define the path where the test images will be saved
test_dir = 'test/images'

# Create the directory if it doesn't exist
os.makedirs(test_dir, exist_ok=True)

# Save the test images to the test/images directory
for idx, img in enumerate(test_images):
    img.save(os.path.join(test_dir, f"test_image_{idx+1}.jpg"))  # Save with a new name

print(f"Test images have been saved to {test_dir}")

Test images have been saved to test/images


##  Inference on Far-Field Test Images using Trained YOLOv8 Model

We perform inference using the **best trained YOLOv8 model** to evaluate its performance on the **test set** of far-field fire scenarios.

- 🎯 **Model Weights**: `best.pt` from the training run
- 💾 **Output**: Predictions will be saved automatically by YOLO

This step visually validates how well the model generalizes to unseen far-field fire detection cases.


In [ ]:
# ------------------------ Test & Metrics ------------------------
model.eval()
TP = FP = FN = 0
ious = []

with torch.no_grad():
    for images, targets in tqdm(test_loader, desc="🧪 Testing"):
        images = torch.stack(images).to(device)
        outputs = model(images)
        pred_boxes = outputs[0]['boxes'].cpu()
        scores = outputs[0]['scores'].cpu()
        gt_boxes = targets[0]['boxes'].cpu()

        keep = scores >= 0.5
        pred_boxes = pred_boxes[keep]

        if len(pred_boxes) > 0 and len(gt_boxes) > 0:
            iou = box_iou(pred_boxes, gt_boxes)
            ious.extend(iou.max(dim=1).values.numpy().tolist())
            for i in range(len(pred_boxes)):
                if iou[i].max().item() >= 0.5:
                    TP += 1
                else:
                    FP += 1
            FN += len(gt_boxes) - TP
        elif len(gt_boxes) > 0:
            FN += len(gt_boxes)
        elif len(pred_boxes) > 0:
            FP += len(pred_boxes)

precision = TP / (TP + FP + 1e-6)
recall = TP / (TP + FN + 1e-6)
f1 = 2 * (precision * recall) / (precision + recall + 1e-6)
avg_iou = np.mean(ious) if ious else 0.0

print(f"\n📊 FarField Metrics:")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 Score:  {f1:.4f}")
print(f"Average IoU: {avg_iou:.4f}")

# ------------------------ Optional Visualization ------------------------
def plot_predictions(image, boxes):
    img = image.permute(1, 2, 0).cpu().numpy()
    img = (img * [0.229, 0.224, 0.225] + [0.485, 0.456, 0.406])
    img = np.clip(img, 0, 1)
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.imshow(img)
    for box in boxes:
        x1, y1, x2, y2 = box
        ax.add_patch(plt.Rectangle((x1, y1), x2-x1, y2-y1,
                                   fill=False, color='red', linewidth=2))
    plt.axis("off"); plt.title("FarField Prediction"); plt.show()

sample_img, _ = test_dataset[0]
model.eval()
with torch.no_grad():
    out = model([sample_img.to(device)])
    plot_predictions(sample_img.cpu(), out[0]['boxes'].cpu().detach())